In [ ]:
import requests

url = "https://appihealthgroup.com/instructor-directory/"
headers = {"User-Agent": "Mozilla/5.0 (BodyOwn directory research)"}
resp = requests.get(url, params={"wpbdp_view": "all_listings"}, headers=headers, timeout=20)

print(resp.status_code)
with open("appi_page1.html", "w", encoding="utf-8") as f:
    f.write(resp.text)

200


In [ ]:
import csv

def save_to_csv(entries, path="appi_raw.csv"):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(entries[0].keys()))
        writer.writeheader()
        writer.writerows(entries)

In [ ]:
import time
import requests
from bs4 import BeautifulSoup

def parse_listing(listing_div):
    entry = {}
    name_tag = listing_div.select_one(".listing-title h3 a")
    h3_name = name_tag.get_text(strip=True) if name_tag else ""
    entry["listing_url"] = name_tag["href"] if name_tag and name_tag.has_attr("href") else ""

    def get_field(field_class):
        field_div = listing_div.find("div", class_=field_class)
        if not field_div:
            return ""
        value_div = field_div.find("div", class_="value") or field_div.find("div")
        return value_div.get_text(" ", strip=True) if value_div else ""

    fallback_name = get_field("wpbdp-field-business_name")
    entry["name"] = h3_name or fallback_name
    entry["name_needs_review"] = not entry["name"]
    entry["description"] = get_field("wpbdp-field-long_business_description")
    entry["website"] = get_field("wpbdp-field-business_website_address")
    entry["phone"] = get_field("wpbdp-field-business_phone_number")

    addr_div = listing_div.find("div", class_="address-info")
    entry["address"] = addr_div.find("div").get_text(" ", strip=True) if addr_div and addr_div.find("div") else ""
    return entry


def parse_page(html):
    soup = BeautifulSoup(html, "html.parser")
    return [parse_listing(l) for l in soup.select("div[id^='wpbdp-listing-']")]


def get_next_page_url(html):
    soup = BeautifulSoup(html, "html.parser")
    next_link = soup.select_one(".wpbdp-pagination .next a")
    return next_link["href"] if next_link else None


def scrape_all_pages(start_url="https://appihealthgroup.com/instructor-directory/", max_pages=None, delay=1.5):
    headers = {"User-Agent": "Mozilla/5.0 (BodyOwn directory research)"}
    all_entries = []
    url = start_url
    page_num = 1

    while url:
        print(f"Fetching page {page_num}: {url}")
        resp = requests.get(url, params={"wpbdp_view": "all_listings"} if page_num == 1 else None,
                             headers=headers, timeout=20)
        if resp.status_code != 200:
            print(f"  Stopped: status {resp.status_code}")
            break

        html = resp.text
        entries = parse_page(html)
        all_entries.extend(entries)
        print(f"  -> {len(entries)} listings (running total: {len(all_entries)})")

        url = get_next_page_url(html)
        page_num += 1
        if max_pages and page_num > max_pages:
            print("Hit max_pages limit, stopping.")
            break
        time.sleep(delay)

    return all_entries


if __name__ == "__main__":
    results = scrape_all_pages(max_pages=None)
    print(f"\nTotal listings scraped: {len(results)}")
    flagged = [e for e in results if e["name_needs_review"]]
    print(f"Flagged for name review: {len(flagged)}")
    save_to_csv(results)
    print("Saved to appi_raw.csv")

Fetching page 1: https://appihealthgroup.com/instructor-directory/
  -> 10 listings (running total: 10)
Fetching page 2: https://appihealthgroup.com/instructor-directory/page/2/?wpbdp_view=all_listings
  -> 10 listings (running total: 20)
Fetching page 3: https://appihealthgroup.com/instructor-directory/page/3/?wpbdp_view=all_listings
  -> 10 listings (running total: 30)
Fetching page 4: https://appihealthgroup.com/instructor-directory/page/4/?wpbdp_view=all_listings
  -> 10 listings (running total: 40)
Fetching page 5: https://appihealthgroup.com/instructor-directory/page/5/?wpbdp_view=all_listings
  -> 10 listings (running total: 50)
Fetching page 6: https://appihealthgroup.com/instructor-directory/page/6/?wpbdp_view=all_listings
  -> 10 listings (running total: 60)
Fetching page 7: https://appihealthgroup.com/instructor-directory/page/7/?wpbdp_view=all_listings
  -> 10 listings (running total: 70)
Fetching page 8: https://appihealthgroup.com/instructor-directory/page/8/?wpbdp_view=a

In [ ]:
!pip install google-generativeai

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = ""

In [ ]:
import requests

resp = requests.get(
    "https://generativelanguage.googleapis.com/v1beta/models",
    params={"key": GEMINI_API_KEY},
)
for m in resp.json().get("models", []):
    if "generateContent" in m.get("supportedGenerationMethods", []):
        print(m["name"])

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
import requests

resp = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent",
    params={"key": GEMINI_API_KEY},
    headers={"Content-Type": "application/json"},
    json={"contents": [{"parts": [{"text": "say hi"}]}]},
)
print(resp.status_code)
print(resp.text)

404
{
  "error": {
    "code": 404,
    "message": "This model models/gemini-2.5-pro is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.",
    "status": "NOT_FOUND"
  }
}



In [ ]:
import csv
import json
import time
import os
import sys
import traceback
import requests

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent"
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

CLASSIFY_PROMPT = """You are checking a Pilates instructor's bio for evidence of pre/postnatal training, for a health directory verification task.

Classify the text into exactly one of these three categories:

- "named_qualification": the text attaches a specific NAME to the pre/postnatal training — an awarding body (e.g. APPI, YMCA, Focus Awards, Active IQ), an institution (e.g. "University of Melbourne"), a level/grade (e.g. "Level 3"), an accreditation number (e.g. Ofqual), or a specific branded course/exam title. The body or specific title must be present in THIS snippet, not just implied by context elsewhere in the bio.
- "claimed_only": the text mentions ante-natal, post-natal, pre-natal, or perinatal Pilates/training as something the instructor does, teaches, or has "completed" or "attended" — but the ONLY thing named is the topic itself (e.g. "Ante and Post Natal Pilates", "Pre and Post Natal", "Antenatal/Postnatal Pilates certification"). Do NOT upgrade this to named_qualification just because the text uses words like "course", "training", "certified", or "certification" — those words alone do not name a body or credential.
- "unverified": no mention of pre/postnatal/perinatal work at all.

CRITICAL DISTINCTION — this is the most common error, so check it carefully:
- "APPI Ante and Post Natal Pilates" -> named_qualification (APPI is a named awarding body)
- "Ante and Post Natal Pilates" (no APPI, no other body) -> claimed_only, even though it sounds like a course name
- "Pre and Postnatal Pilates certification" -> claimed_only (claims a certificate exists but does not say who issued it or what it's called)
- "Fully Certified Ante and Postnatal Instructor" -> claimed_only (claims certified status but names no body)
- "Level 3 Pre & Post-Natal Exercise" -> named_qualification (a specific accreditation level/title)
- "Master's Degree in Physiotherapy in Women's Health" -> named_qualification (a specific degree)

Rule of thumb: ask "if I removed the words ante/post/pre-natal, would there still be a proper noun, level, or institution left in this snippet?" If yes -> named_qualification. If the snippet would just disappear -> claimed_only.

Also separately flag:
- "is_physio": true if the text indicates the person is a registered/chartered physiotherapist (e.g. mentions "physiotherapist", "MCSP", "HCPC", "chartered physio"), false otherwise.
- "is_appi_pregnancy_postnatal_cert": true ONLY if the named qualification is specifically the APPI Pregnancy/Antenatal/Postnatal Pilates certification (any wording variant of "APPI Ante/Post Natal Pilates" counts). false if the named qualification is something else — even if clearly perinatal-adjacent — such as a general Women's Health course/degree, a Pelvic Floor Physiotherapy certificate, an Obstetrics & Gynaecology PT course, or any non-APPI-branded award. This must be false whenever layer2_status is not "named_qualification".
- "has_exclusion": true if the text mentions treating diastasis recti or pelvic floor issues (clinical/physio scope, not fitness scope), false otherwise.

Respond ONLY with valid JSON, no markdown, no explanation:
{"layer2_status": "...", "matched_snippet": "...", "is_physio": true/false, "is_appi_pregnancy_postnatal_cert": true/false, "has_exclusion": true/false}

If layer2_status is "unverified", matched_snippet should be an empty string.

TEXT TO CLASSIFY:
"""

# Seconds to wait for a single Gemini call before giving up and retrying.
# Without this, a stalled request hangs forever with no error and no timeout -
# which is what caused the multi-hour "stuck on row 1" issue.
API_TIMEOUT_SECONDS = 60

EMPTY_RESULT = {
    "layer2_status": "unverified",
    "matched_snippet": "",
    "is_physio": False,
    "is_appi_pregnancy_postnatal_cert": False,
    "has_exclusion": False,
}

ERROR_RESULT = {
    "layer2_status": "error",
    "matched_snippet": "",
    "is_physio": False,
    "is_appi_pregnancy_postnatal_cert": False,
    "has_exclusion": False,
}


def classify_with_gemini(text, retries=3, timeout_seconds=API_TIMEOUT_SECONDS):
    if not text.strip():
        return dict(EMPTY_RESULT)

    headers = {"Content-Type": "application/json"}
    params = {"key": GEMINI_API_KEY}
    payload = {"contents": [{"parts": [{"text": CLASSIFY_PROMPT + text}]}]}

    last_raw = None
    for attempt in range(retries):
        try:
            print(f"    [DEBUG] sending request to Gemini (attempt {attempt+1}/{retries}, timeout={timeout_seconds}s)...", file=sys.stderr, flush=True)
            t0 = time.monotonic()
            resp = requests.post(
                GEMINI_URL,
                headers=headers,
                params=params,
                json=payload,
                timeout=timeout_seconds,
            )
            elapsed = time.monotonic() - t0
            print(f"    [DEBUG] got response back from Gemini in {elapsed:.1f}s (HTTP {resp.status_code})", file=sys.stderr, flush=True)
            resp.raise_for_status()
            data = resp.json()
            raw = data["candidates"][0]["content"]["parts"][0]["text"].strip()
            last_raw = raw
            # Strip markdown code fences if the model adds them despite instructions
            raw = raw.replace("```json", "").replace("```", "").strip()
            result = json.loads(raw)
            return result
        except Exception as e:
            print(f"    [ERROR] attempt {attempt+1}/{retries} failed: {type(e).__name__}: {e}", file=sys.stderr, flush=True)
            if last_raw is not None:
                print(f"    [ERROR] raw model output was: {last_raw[:300]!r}", file=sys.stderr, flush=True)
            else:
                print("    [ERROR] no usable response received from API call (timeout/network/auth/rate-limit/safety-block?)", file=sys.stderr, flush=True)
            if attempt < retries - 1:
                print(f"    [INFO] retrying in {2 ** attempt}s...", file=sys.stderr, flush=True)
                time.sleep(2 ** attempt)  # backoff: 1s, 2s, 4s
                continue
            print(f"  [WARN] Classification failed after {retries} attempts: {e}", file=sys.stderr, flush=True)
            traceback.print_exc()
            return dict(ERROR_RESULT)


def preflight_check():
    """Run one real classification call before touching the CSV, so a broken
    connection fails in seconds instead of silently eating hours across 570 rows."""
    print("[INFO] Running preflight check against Gemini API...", flush=True)
    test = classify_with_gemini(
        "Pilates instructor, no specific qualifications listed.",
        retries=1,
        timeout_seconds=15,
    )
    if test.get("layer2_status") == "error":
        print("[FATAL] Preflight check failed — API is unreachable or misconfigured. "
              "Aborting before processing the full CSV.", file=sys.stderr)
        sys.exit(1)
    print(f"[INFO] Preflight check passed (got: {test}).", flush=True)


def verify_entry(entry):
    text = f"{entry.get('description', '')} {entry.get('address', '')}"
    result = classify_with_gemini(text)

    layer2_status = result.get("layer2_status", "error")
    is_physio = result.get("is_physio", False)
    is_appi_cert = result.get("is_appi_pregnancy_postnatal_cert", False)
    has_exclusion = result.get("has_exclusion", False)

    is_priority = is_physio and layer2_status == "named_qualification" and is_appi_cert

    entry["layer2_status"] = layer2_status
    entry["matched_snippet"] = result.get("matched_snippet", "")
    entry["is_physio"] = is_physio
    entry["is_appi_pregnancy_postnatal_cert"] = is_appi_cert
    entry["is_appi_physio"] = is_priority
    entry["tier"] = "needs_manual_check" if has_exclusion else ("priority" if is_priority else "standard")
    entry["exclusion_flag"] = has_exclusion
    entry["ofqual_verified"] = ""  # blank for your manual registry check

    # Visibility into what each row actually resolved to
    print(f"    -> status={layer2_status} tier={entry['tier']} "
          f"physio={is_physio} appi_cert={is_appi_cert} exclusion={has_exclusion}", flush=True)

    return entry


def load_and_verify(path="appi_raw.csv", delay=1.0):
    print(f"[INFO] Opening {path}...", flush=True)
    try:
        with open(path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        print(f"[FATAL] Could not find {path} — check the working directory / filename.", file=sys.stderr)
        raise
    except Exception:
        print(f"[FATAL] Failed to read {path}:", file=sys.stderr)
        traceback.print_exc()
        raise

    print(f"[INFO] Loaded {len(rows)} rows from {path}", flush=True)
    if not rows:
        print("[WARN] CSV has 0 data rows — nothing to classify.", file=sys.stderr)

    verified = []
    for i, row in enumerate(rows):
        print(f"Classifying {i+1}/{len(rows)}: {row.get('name', '')[:40]}", flush=True)
        try:
            verified.append(verify_entry(row))
        except Exception:
            # Don't let one bad row silently kill the whole run without a trace
            print(f"  [FATAL] Unhandled exception on row {i+1} ({row.get('name','')[:40]}):", file=sys.stderr)
            traceback.print_exc()
            row["layer2_status"] = "error"
            row["matched_snippet"] = ""
            row["is_physio"] = False
            row["is_appi_pregnancy_postnatal_cert"] = False
            row["is_appi_physio"] = False
            row["tier"] = "standard"
            row["exclusion_flag"] = False
            row["ofqual_verified"] = ""
            verified.append(row)
        time.sleep(delay)  # stay under rate limits

    return verified


def save_verified(entries, path="appi_verified.csv"):
    if not entries:
        print("[WARN] No entries to save — skipping CSV write.", file=sys.stderr)
        return
    print(f"[INFO] Writing {len(entries)} rows to {path}...", flush=True)
    try:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(entries[0].keys()))
            writer.writeheader()
            writer.writerows(entries)
    except Exception:
        print(f"[FATAL] Failed to write {path}:", file=sys.stderr)
        traceback.print_exc()
        raise
    print(f"[INFO] Successfully wrote {path}", flush=True)


if __name__ == "__main__":
    preflight_check()

    entries = load_and_verify()

    named = [e for e in entries if e["layer2_status"] == "named_qualification"]
    claimed = [e for e in entries if e["layer2_status"] == "claimed_only"]
    unverified = [e for e in entries if e["layer2_status"] == "unverified"]
    errors = [e for e in entries if e["layer2_status"] == "error"]
    priority = [e for e in entries if e["tier"] == "priority"]
    needs_review = [e for e in entries if e["tier"] == "needs_manual_check"]
    excluded = [e for e in entries if e["exclusion_flag"]]

    print(f"\nTotal: {len(entries)}")
    print(f"Named qualification: {len(named)}")
    print(f"Claimed only: {len(claimed)}")
    print(f"Unverified: {len(unverified)}")
    print(f"Classification errors: {len(errors)}")
    print(f"Priority tier: {len(priority)}")
    print(f"Needs manual check (exclusion): {len(needs_review)}")
    print(f"Exclusion flagged: {len(excluded)}")

    save_verified(entries, "appi_verified.csv")
    print("\nSaved appi_verified.csv")

[INFO] Running preflight check against Gemini API...


    [DEBUG] sending request to Gemini (attempt 1/1, timeout=15s)...
    [DEBUG] got response back from Gemini in 3.6s (HTTP 200)


[INFO] Preflight check passed (got: {'layer2_status': 'unverified', 'matched_snippet': '', 'is_physio': False, 'is_appi_pregnancy_postnatal_cert': False, 'has_exclusion': False}).
[INFO] Opening appi_raw.csv...
[INFO] Loaded 570 rows from appi_raw.csv
Classifying 1/570: 


    [DEBUG] sending request to Gemini (attempt 1/3, timeout=60s)...
    [DEBUG] got response back from Gemini in 7.3s (HTTP 200)


    -> status=named_qualification tier=standard physio=False appi_cert=True exclusion=False
Classifying 2/570: *APPI Clinics Hampstead


...

    -> status=unverified tier=standard physio=False appi_cert=False exclusion=False
Classifying 569/570: Zafeiro Sverkou (Matwork Certified Instr


    [DEBUG] sending request to Gemini (attempt 1/3, timeout=60s)...
    [DEBUG] got response back from Gemini in 4.8s (HTTP 200)


    -> status=unverified tier=standard physio=True appi_cert=False exclusion=False
Classifying 570/570: Zoe Rex (Fully Certified)


    [DEBUG] sending request to Gemini (attempt 1/3, timeout=60s)...
    [DEBUG] got response back from Gemini in 5.6s (HTTP 200)


    -> status=unverified tier=standard physio=True appi_cert=False exclusion=False

Total: 570
Named qualification: 154
Claimed only: 70
Unverified: 346
Classification errors: 0
Priority tier: 60
Needs manual check (exclusion): 29
Exclusion flagged: 29
[INFO] Writing 570 rows to appi_verified.csv...
[INFO] Successfully wrote appi_verified.csv

Saved appi_verified.csv


In [ ]:
import csv

with open('appi_verified.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

named = [r for r in rows if r['layer2_status'] == 'named_qualification']

unique_snippets = sorted(set(r['matched_snippet'].strip() for r in named))

print(f"{len(unique_snippets)} unique values out of {len(named)} rows")
for s in unique_snippets:
    print(s)

145 unique values out of 154 rows
APPI Ante and Post Natal Pilates
APPI Ante and Post natal Pilates
APPI Ante and Postnatal Pilates
APPI Ante-Natal and Post-Natal Pilates
APPI Ante/Post Natal Pilates
APPI Ante/Postnatal Pilates
APPI Antenatal and Postnatal Pilates
APPI Antenatal and Postnatal qualifications in Matwork and Reformer Pilates
APPI COURSES Matwork level one Matwork level two ( class instructor) Matwork level three Pilates for Osteoporosis Ante/post natal Pilates
APPI CPD Ante & Post-Natal Course
APPI Certified Pilates Matwork Instructor APPI Certified Pilates Equipment Instructor Courses: Matwork Level 1 Matwork Level 2 Matwork Level 3 Equipment Level 1 Equipment Level 2 Equipment Level 3 Equipment Level 4 Pilates for Rehabilitation Osteoporosis Pilates for Rehabilitation Ante/Post Natal
APPI Comprehensive Course Ante and Post Natal
APPI Courses : Fully Certified Matwork Instructor Matwork Level One Matwork Level Two Matwork Level Three - Ante / Post Natal Pilates
APPI Cour

In [ ]:
import pandas as pd

# Input/output paths — adjust if needed
INPUT_CSV = "appi_verified.csv"
OUTPUT_CSV = "triage_false_triggers.csv"

# The rows to pull out for manual review
TRIAGE_NAMES = [
    "Elisa Gallippi (Fully Certified)",
    "Jenny Gebka (Fully Certified)",
    "Piotr Peter Kardas (Fully Certified)",
]

df = pd.read_csv(INPUT_CSV).fillna("")

triage = df[df["name"].isin(TRIAGE_NAMES)].copy()

# Save a standalone CSV with full, untruncated fields for manual review
triage.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(triage)} rows to {OUTPUT_CSV}")

# Print full bios to the console for a quick look
for _, row in triage.iterrows():
    print("=" * 80)
    print("NAME:", row["name"])
    print("URL:", row["listing_url"])
    print("STATUS:", row["layer2_status"], "| exclusion_flag:", row["exclusion_flag"])
    print("FULL DESCRIPTION:")
    print(row["description"])
    print()

Saved 3 rows to triage_false_triggers.csv
NAME: Elisa Gallippi (Fully Certified)
URL: https://appihealthgroup.com/instructor-directory/1232278/elisa-gallippi-fully-certified/
STATUS: unverified | exclusion_flag: True
FULL DESCRIPTION:
• Name: Elisa Gallippi • APPI courses that you have attended: Full Matwork; Kids and Teens; Reformer with resistance tubes • Pilates classes that you run (times and days): Details will be on the website • 1:1 Sessions (times and days): Every day, by appointment only • Area’s that you specialize in e.g. ante/post-natal: •: I specialize in Pilates for sportspeople and athletes, including runners, skiers, rock climbers, and skaters. Additionally, I have extensive expertise in diastasis recti and provide customized one-on-one sessions for individuals who currently have or have previously experienced it.

NAME: Jenny Gebka (Fully Certified)
URL: https://appihealthgroup.com/instructor-directory/1018935/1018935/
STATUS: unverified | exclusion_flag: True
FULL DES

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('appi_verified (5).csv')

# Get an array of unique address values (includes NaN/missing values if present)
unique_addresses = df['address'].unique()

# Print the unique values
print(unique_addresses)

['Hinckley, Leicestershire'
 '2B Heath Hurst Road Hampstead NW3 2RX NW3 2RX'
 '1 Thornton Road Wimbledon Village SW19 4NB SW19 4NB'
 'Stoke Gifford South Gloucestershire' nan
 '230 Burlington Road New Malden KT3 4NW'
 'Surbiton New Life Baptist Church 6 Langley Road, KT6 6LN' 'Gower'
 '40 Caswell Road, Swansea SA3 4SD' '53 Halton road Carterton OX18 3SD'
 'Cirencester Pilates, Old Station Works, Sheep Street, Cirencester, GL7 1RQ'
 'Indigo House Medical Centre, 2nd Floor, 2-8 Oxford Road, St Helier, JE1 4HB Jersey, Channel Island'
 'Cape Town, South Africa'
 'Enliven Health ltd Unit, 3 5 West Hill, Aspley Guise MK178DP'
 'Performance Physiotherapy Ltd, Indigo House, 2-8 Oxford Rd, St Helier, Jersey, JE3 5AL'
 'Gilberd Road Colchester CO2 7LR' '22116 180th Ave, Lewistown, MO 63452'
 '69 Osbourne Road, Windsor, SL4 3EQ' '247 Smith Road, Grand Cayman'
 'Eglinton Wellness Centre, 3A1 Benbow Industrial Estate, 15 Killylane Road, Londonderry BT47 3DW'
 'Amy McKeen, PT DPT Courage Kenny Sport

In [ ]:
import csv
import json
import os
import sys
import time
import requests

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent"
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

REGION_PROMPT = """Extract the primary administrative area from the following address.

You MUST return a single CANONICAL name for the real-world place — never copy misspellings,
abbreviations, or ad-hoc phrasing from the source address. If you already know the correct
official name of the place, use it, even if the address text spells it differently.

RULES BY COUNTRY:

1. UK, Greater London addresses -> return the specific LONDON BOROUGH (e.g. "Camden",
   "Wandsworth", "Islington", "Merton", "Tower Hamlets", "Hackney"), NOT "London" and NOT the
   neighborhood/area name. Use your knowledge of UK geography to map any neighborhood, area, or
   postcode (NW, SW, SE, E, EC, W, WC, N prefixes) to its correct borough. Examples: Hampstead ->
   Camden, Wimbledon -> Merton, Wapping -> Tower Hamlets, Croydon -> Croydon, Ilford -> Redbridge.

2. UK, outside Greater London -> return the ceremonial COUNTY (e.g. "Essex", "Leicestershire",
   "Berkshire"), not a specific town, UNLESS the town/city is itself a well-known standalone
   unitary authority commonly referred to by its own name (e.g. "Bristol", "Manchester",
   "Birmingham" stay as-is). Always use the standard modern county name/spelling (e.g.
   "Gloucestershire" not "Gloustershire").

3. Ireland (Republic or Northern Ireland) -> always return "County X" for the county-level unit
   (e.g. "County Kerry", "County Antrim", "County Down"), never just "X" on its own, and never a
   town name in place of the county, unless the address is within Dublin, Belfast, or Cork city
   proper, in which case return the city name ("Dublin", "Belfast", "Cork").

4. Germany -> always return the Bundesland (federal state) in its standard English name, even if
   the address does not mention it explicitly and even if it's misspelled in the source address —
   infer it from the city/postcode using your knowledge of German geography. Never return a city
   or town name for Germany. Use these exact English names: "Baden-Württemberg", "Bavaria",
   "Berlin", "Brandenburg", "Bremen", "Hamburg", "Hesse", "Lower Saxony", "Mecklenburg-Vorpommern",
   "North Rhine-Westphalia", "Rhineland-Palatinate", "Saarland", "Saxony", "Saxony-Anhalt",
   "Schleswig-Holstein", "Thuringia". (So "Niedersachsen", "Neidersachen", "Niedersachen" all ->
   "Lower Saxony"; "Sachsen" -> "Saxony"; "Hessen" -> "Hesse".)

5. Other non-UK/Ireland/Germany addresses -> return the country's primary state/province/region
   using its standard English name (e.g. "California", "Bavaria"-style state, "Attica" for
   Athens, "New South Wales" not "NSW"). If no state/province concept applies, return the city
   (e.g. "Singapore", "Dubai", "Minneapolis" -> actually return "Minnesota" if a US state exists;
   only fall back to city for city-states or when no broader region exists).

6. Always expand abbreviations to full standard names (e.g. "NSW" -> "New South Wales", "Bucks"
   -> "Buckinghamshire", "Herts" -> "Hertfordshire", "Notts" -> "Nottinghamshire").

7. If no region can be determined at all, return an empty string "".

Respond ONLY with valid JSON:
{"region": "..."}

ADDRESS TO EXTRACT FROM:
"""


def extract_region_with_gemini(address, retries=3):
  if not address or not address.strip():
    return ""

  headers = {"Content-Type": "application/json"}
  params = {"key": GEMINI_API_KEY}
  payload = {"contents": [{"parts": [{"text": REGION_PROMPT + address}]}]}

  for attempt in range(retries):
    try:
      resp = requests.post(
          GEMINI_URL, headers=headers, params=params, json=payload, timeout=30
      )
      resp.raise_for_status()
      data = resp.json()

      raw = data["candidates"][0]["content"]["parts"][0]["text"].strip()
      raw = raw.replace("```json", "").replace("```", "").strip()

      result = json.loads(raw)
      return result.get("region", "")
    except Exception as e:
      if attempt < retries - 1:
        time.sleep(1)
        continue
      print(
          f"  [WARN] Failed to get region for '{address[:30]}...': {e}",
          file=sys.stderr,
      )
      return ""


def process_csv(
    input_file="appi_verified (5).csv", output_file="appi_with_region.csv"
):
  print(f"[INFO] Reading {input_file}...", flush=True)

  with open(input_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

  print(
      f"[INFO] Loaded {len(rows)} rows. Extracting regions...", flush=True
  )

  for i, row in enumerate(rows):
    address = row.get("address", "")
    region = extract_region_with_gemini(address)
    row["region"] = region

    print(
        f"[{i+1}/{len(rows)}] '{address[:30]}...' -> region: '{region}'",
        flush=True,
    )
    time.sleep(0.5)  # Rate-limit safety pause

  # Save updated CSV
  with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

  print(f"\n[SUCCESS] Done! Saved output with 'region' column to {output_file}")


if __name__ == "__main__":
  process_csv()

[INFO] Reading appi_verified (5).csv...
[INFO] Loaded 570 rows. Extracting regions...
[1/570] 'Hinckley, Leicestershire...' -> region: 'Leicestershire'
[2/570] '2B Heath Hurst Road Hampstead ...' -> region: 'Camden'
[3/570] '1 Thornton Road Wimbledon Vill...' -> region: 'Merton'
[4/570] 'Stoke Gifford South Gloucester...' -> region: 'Gloucestershire'
[5/570] '...' -> region: ''
[6/570] '230 Burlington Road New Malden...' -> region: 'Kingston upon Thames'
[7/570] 'Surbiton New Life Baptist Chur...' -> region: 'Kingston upon Thames'
[8/570] 'Gower...' -> region: 'Swansea'
[9/570] '40 Caswell Road, Swansea SA3 4...' -> region: 'Swansea'
[10/570] '53 Halton road Carterton OX18 ...' -> region: 'Oxfordshire'
[11/570] 'Cirencester Pilates, Old Stati...' -> region: 'Gloucestershire'
[12/570] 'Indigo House Medical Centre, 2...' -> region: 'Jersey'
[13/570] 'Cape Town, South Africa...' -> region: 'Western Cape'
[14/570] 'Enliven Health ltd Unit, 3 5 W...' -> region: 'Bedfordshire'
[15/570] 'Perf

In [ ]:
import csv

INPUT_FILE = "appi_with_region.csv"          # output from the Gemini region script
OUTPUT_FILE = "appi_london_essex_final.csv"   # confirmed London/Essex entries
VAGUE_FILE = "appi_vague_london_check.csv"    # blank region but "London"/"Essex" mentioned in address

LONDON_BOROUGHS = {
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley", "Camden",
    "Croydon", "Ealing", "Enfield", "Greenwich", "Hackney",
    "Hammersmith and Fulham", "Haringey", "Harrow", "Havering", "Hillingdon",
    "Hounslow", "Islington", "Kensington and Chelsea", "Kingston upon Thames",
    "Lambeth", "Lewisham", "Merton", "Newham", "Redbridge",
    "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
    "Waltham Forest", "Wandsworth", "Westminster", "City of London",
}


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = rows[0].keys()

    confirmed = [
        r for r in rows
        if r["region"] in LONDON_BOROUGHS or r["region"] == "Essex"
    ]

    vague = [
        r for r in rows
        if not r["region"].strip()
        and r["address"].strip()
        and ("london" in r["address"].lower() or "essex" in r["address"].lower())
    ]

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(confirmed)

    with open(VAGUE_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(vague)

    print(f"Confirmed London/Essex: {len(confirmed)} -> {OUTPUT_FILE}")
    print(f"Needs manual check (vague London/Essex mention, no borough): {len(vague)} -> {VAGUE_FILE}")


if __name__ == "__main__":
    main()

Confirmed London/Essex: 35 -> appi_london_essex_final.csv
Needs manual check (vague London/Essex mention, no borough): 2 -> appi_vague_london_check.csv


In [ ]:
import csv

INPUT_FILE = "appi_london_essex_final.csv"
OUTPUT_FILE = "appi_london_essex_verified_only.csv"

DROP_STATUSES = {"unverified", "claimed_only"}


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = rows[0].keys()

    kept = [r for r in rows if r["layer2_status"] not in DROP_STATUSES]

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(kept)

    print(f"Kept {len(kept)} of {len(rows)} rows -> {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Kept 10 of 35 rows -> appi_london_essex_verified_only.csv


In [ ]:
import csv

INPUT_FILE = "/content/appi_london_essex_verified_only.csv"
OUTPUT_FILE = "appi_ready_for_cimspa_check.csv"  # overwrites in place

# Add one entry per person you've verified. Only include the fields you want
# to set - leave others out and they won't be touched.
UPDATES = {
    "Katrina Wade": {
        "postcode": "CO4 9AS",
        "hcpc_registered": "Yes",
        "hcpc_number": "PH32618",
        "cimspa_member": "N/A - HCPC registered physiotherapist",
    },
    # "Aga Hanusiak": {
    #     "hcpc_registered": "Yes",
    #     "hcpc_number": "PHxxxxx",
    #     "cimspa_member": "N/A - HCPC registered physiotherapist",
    # },
}


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = list(rows[0].keys())

    # make sure any new fields referenced in UPDATES exist as columns
    for updates in UPDATES.values():
        for field in updates:
            if field not in fieldnames:
                fieldnames.append(field)

    updated_count = 0
    for row in rows:
        for name_key, updates in UPDATES.items():
            if name_key in row["name"]:
                row.update(updates)
                updated_count += 1

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"Updated {updated_count} row(s) -> {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Updated 1 row(s) -> appi_ready_for_cimspa_check.csv


In [ ]:
import csv
import re

INPUT_FILE = "appi_ready_for_cimspa_check.csv"
OUTPUT_FILE = "appi_ready_for_cimspa_check.csv"  # overwrites in place

POSTCODES = {
    "Aga Hanusiak": "KT3 4NW",
    "Agnieszka Waszkielis": "KT6 6LN",
    "Emilie Askew": "CO2 8AQ",
    "Helen Ainsworth": "DA5 3JB",
    "Helen Pearce": "WC1N 3BG",
    "Jehian Yehia": "KT2 6HR",
    "Jennifer Dennis": "SW4 9AU",
    "Joanne Murphy": "SE9 2SY",
    "Katrina Wade": "CO4 9AS",
    "Noemi Nagy": "EN4 8HG",
}


def normalize(text):
    """Strip parentheses/brackets and collapse whitespace/case so matching
    is resilient to formatting like '(Aga)' or extra spacing."""
    text = re.sub(r"[()\[\]]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = list(rows[0].keys())

    if "postcode" not in fieldnames:
        fieldnames.append("postcode")

    normalized_postcodes = {
        normalize(name_key): (name_key, postcode)
        for name_key, postcode in POSTCODES.items()
    }

    updated_count = 0
    matched_keys = set()
    for row in rows:
        norm_name = normalize(row["name"])
        for norm_key, (name_key, postcode) in normalized_postcodes.items():
            if norm_key in norm_name:
                row["postcode"] = postcode
                updated_count += 1
                matched_keys.add(name_key)

    unmatched = set(POSTCODES.keys()) - matched_keys
    if unmatched:
        print(f"WARNING: no row matched for: {sorted(unmatched)}")

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"Updated {updated_count} row(s) -> {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Updated 10 row(s) -> appi_ready_for_cimspa_check.csv


In [ ]:
import csv

INPUT_FILE = "appi_ready_for_cimspa_check.csv"
OUTPUT_FILE = "appi_ready_for_cimspa_check.csv"  # overwrites in place

UPDATES = {
    "Helen Ainsworth": {
        "hcpc_registered": "Yes",
        "hcpc_number": "PH81945",
        "cimspa_member": "N/A - HCPC registered physiotherapist",
    },
    "Aga Hanusiak": {
        "hcpc_registered": "No match under Hanusiak - bio claims HCPC physio, needs manual follow-up (possible maiden name or unverified claim)",
        "hcpc_number": "",
    },
}


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = list(rows[0].keys())

    for updates in UPDATES.values():
        for field in updates:
            if field not in fieldnames:
                fieldnames.append(field)

    updated_count = 0
    for row in rows:
        for name_key, updates in UPDATES.items():
            if name_key in row["name"]:
                row.update(updates)
                updated_count += 1

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"Updated {updated_count} row(s) -> {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Updated 1 row(s) -> appi_ready_for_cimspa_check.csv


In [ ]:
import csv
import re

INPUT_FILE = "appi_ready_for_cimspa_check.csv"
OUTPUT_FILE = "appi_ready_for_cimspa_check.csv"  # overwrites in place

UPDATES = {
    "Aga Hanusiak": {
        "hcpc_registered": "Yes",
        "hcpc_number": "PH100782",
        "cimspa_member": "N/A - HCPC registered physiotherapist",
    },

}


def normalize(text):
    """Strip parentheses/brackets and collapse whitespace/punctuation so
    matching is resilient to formatting like '(Aga)' or extra spacing."""
    text = re.sub(r"[()\[\]]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = list(rows[0].keys())

    for updates in UPDATES.values():
        for field in updates:
            if field not in fieldnames:
                fieldnames.append(field)

    normalized_updates = {
        normalize(name_key): (name_key, updates)
        for name_key, updates in UPDATES.items()
    }

    updated_count = 0
    for row in rows:
        norm_name = normalize(row["name"])
        for norm_key, (name_key, updates) in normalized_updates.items():
            if norm_key in norm_name:
                row.update(updates)
                updated_count += 1

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"Updated {updated_count} row(s) -> {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Updated 1 row(s) -> appi_ready_for_cimspa_check.csv


In [ ]:
# -*- coding: utf-8 -*-
"""
Geocode the postcode column in appi_london_essex_verified_only.csv using
postcodes.io's free bulk lookup endpoint (no API key needed).

postcodes.io allows up to 100 postcodes per bulk request, so this batches
rows in chunks of 100 rather than hitting the API once per row.
"""

import csv
import sys
import time
import requests

INPUT_FILE = "/content/appi_ready_for_cimspa_check.csv"
OUTPUT_FILE = "appi_geocoded.csv"
BULK_URL = "https://api.postcodes.io/postcodes"
BATCH_SIZE = 100  # postcodes.io bulk lookup max per request


def clean_postcode(pc):
    """Basic cleanup: strip whitespace, uppercase. Returns '' if empty."""
    if not pc:
        return ""
    return pc.strip().upper()


def bulk_lookup(postcodes, retries=3):
    """
    postcodes.io bulk endpoint: POST {"postcodes": [...]}
    Returns a dict mapping original postcode string -> {"lat":..., "lng":...}
    or None if that postcode couldn't be resolved.
    """
    payload = {"postcodes": postcodes}
    for attempt in range(retries):
        try:
            resp = requests.post(BULK_URL, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            results = {}
            for item in data.get("result", []):
                query = item.get("query")
                res = item.get("result")
                if res:
                    results[query] = {
                        "lat": res.get("latitude"),
                        "lng": res.get("longitude"),
                    }
                else:
                    results[query] = None
            return results
        except Exception as e:
            print(f"  [WARN] bulk lookup attempt {attempt+1}/{retries} failed: {e}",
                  file=sys.stderr)
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            print("  [ERROR] giving up on this batch — leaving these rows blank", file=sys.stderr)
            return {}


def main():
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = list(rows[0].keys())

    for field in ("latitude", "longitude", "geocode_status"):
        if field not in fieldnames:
            fieldnames.append(field)

    # Only attempt geocoding for rows that actually have a postcode
    to_lookup = []
    for row in rows:
        pc = clean_postcode(row.get("postcode", ""))
        row["postcode"] = pc
        if pc:
            to_lookup.append(pc)

    print(f"[INFO] {len(rows)} total rows, {len(to_lookup)} have a postcode to geocode")

    # De-dupe postcodes across rows so we don't waste lookups
    unique_postcodes = list(dict.fromkeys(to_lookup))
    resolved = {}

    for i in range(0, len(unique_postcodes), BATCH_SIZE):
        batch = unique_postcodes[i:i + BATCH_SIZE]
        print(f"[INFO] Looking up batch {i // BATCH_SIZE + 1} "
              f"({len(batch)} postcodes)...")
        batch_results = bulk_lookup(batch)
        resolved.update(batch_results)
        time.sleep(0.5)  # be polite to the free API

    matched = 0
    unmatched = []
    for row in rows:
        pc = row["postcode"]
        if not pc:
            row["latitude"] = ""
            row["longitude"] = ""
            row["geocode_status"] = "no_postcode"
            continue

        result = resolved.get(pc)
        if result:
            row["latitude"] = result["lat"]
            row["longitude"] = result["lng"]
            row["geocode_status"] = "ok"
            matched += 1
        else:
            row["latitude"] = ""
            row["longitude"] = ""
            row["geocode_status"] = "not_found"
            unmatched.append((row.get("name", ""), pc))

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"\n[SUCCESS] Geocoded {matched}/{len(to_lookup)} postcodes -> {OUTPUT_FILE}")
    if unmatched:
        print(f"[WARN] {len(unmatched)} postcode(s) not found by postcodes.io:")
        for name, pc in unmatched:
            print(f"    {name}: '{pc}'")


if __name__ == "__main__":
    main()

[INFO] 10 total rows, 10 have a postcode to geocode
[INFO] Looking up batch 1 (10 postcodes)...

[SUCCESS] Geocoded 10/10 postcodes -> appi_geocoded.csv


In [ ]:
# -*- coding: utf-8 -*-
"""
Fills in the remaining brief fields (business_name, base_qualification,
delivery_mode, insurance_confirmed) and generates a public-facing bio,
in a single Gemini call per row.

Run this in Colab, not in a network-restricted sandbox.
"""

import csv
import json
import os
import sys
import time
import traceback
import requests

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent"
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]  # set this in Colab, don't hardcode it

INPUT_FILE = "appi_geocoded.csv"
OUTPUT_FILE = "appi_enriched.csv"
API_TIMEOUT_SECONDS = 60

ENRICH_PROMPT = """You are enriching a Pilates instructor's directory entry for a perinatal health
directory. You are given their raw bio text, address, and website. Extract the following fields
and write a rewritten public-facing bio.

ABSOLUTE RULE — READ THIS FIRST: You may only use facts that are explicitly present in the
SOURCE DATA below. If a field's answer is not stated in the source, you MUST leave it empty
(or "unknown" where specified) rather than infer, guess, assume, or fill in anything that sounds
plausible. This applies to every field, including the bio. Never pad a thin bio with generic
filler that implies unstated facts (e.g. do not say "fully insured" unless insurance is actually
mentioned; do not say "years of experience" unless a duration is stated; do not assume a
location, class type, or clientele that isn't named). If the source text is sparse, the output
should be short and sparse too — a short, accurate bio is correct; a longer bio filled with
invented specifics is not.

Fields to extract:

- "business_name": the name of their business/studio, ONLY if the source text explicitly names
  one distinct from the person's own name (e.g. "Berrylands Pilates"). Empty string if they only
  practise under their own name or no business name is mentioned. Do not invent one from the
  website domain or address.

- "base_qualification": the Layer 1 base Pilates/fitness teaching qualification mentioned in the
  text — the foundational teaching certification (NOT the pre/postnatal specialism). For APPI
  instructors this is usually their APPI Matwork level(s) (e.g. "APPI Matwork Level One, Two and
  Three" or "APPI Certified Matwork Instructor"). If a physiotherapy qualification is mentioned
  (HCPC/MCSP/Chartered Physiotherapist), include that too since it functions as their base
  credential. If nothing is stated, return an empty string — do not guess a plausible-sounding
  qualification.

- "delivery_mode": one of "in_person", "online", "both", or "unknown" based on whether the text
  mentions home visits, studio classes, online/virtual sessions, or a mix. Default to "unknown"
  if there's no clear signal either way — do not guess.

- "insurance_confirmed": always return "unknown" — this cannot be determined from bio text and
  must be manually verified against public liability insurance records. Do not attempt to infer
  this from certifications or physiotherapy registration, and never write "yes" here regardless
  of how confident the bio sounds.

- "bio": a rewritten, polished, public-facing bio of 60-100 words, professional in tone, written
  in third person, suitable for a health directory listing. Every claim in the bio must trace back
  to something explicitly stated in the source text. Do not invent qualifications, locations,
  years of experience, specialisms, insurance status, or claims not in the original. Do not use
  superlatives not supported by the source (e.g. don't call someone "the best" or "leading" unless
  the source says so). If the source only gives a couple of concrete facts, write a short bio
  built only from those facts rather than stretching it to 60-100 words with invented detail —
  going under the word count is fine, inventing content to hit it is not.

Respond ONLY with valid JSON, no markdown, no explanation:
{"business_name": "...", "base_qualification": "...", "delivery_mode": "...", "insurance_confirmed": "unknown", "bio": "..."}

SOURCE DATA:
"""


EMPTY_RESULT = {
    "business_name": "",
    "base_qualification": "",
    "delivery_mode": "unknown",
    "insurance_confirmed": "unknown",
    "bio": "",
}

ERROR_RESULT = dict(EMPTY_RESULT, bio="[ENRICHMENT FAILED - needs manual bio]")


def enrich_with_gemini(name, description, address, website, retries=3):
    source_text = (
        f"Name: {name}\n"
        f"Address: {address}\n"
        f"Website: {website}\n"
        f"Raw bio text: {description}"
    )

    headers = {"Content-Type": "application/json"}
    params = {"key": GEMINI_API_KEY}
    payload = {"contents": [{"parts": [{"text": ENRICH_PROMPT + source_text}]}]}

    last_raw = None
    for attempt in range(retries):
        try:
            resp = requests.post(
                GEMINI_URL, headers=headers, params=params, json=payload,
                timeout=API_TIMEOUT_SECONDS,
            )
            resp.raise_for_status()
            data = resp.json()
            raw = data["candidates"][0]["content"]["parts"][0]["text"].strip()
            last_raw = raw
            raw = raw.replace("```json", "").replace("```", "").strip()
            return json.loads(raw)
        except Exception as e:
            print(f"    [ERROR] attempt {attempt+1}/{retries} failed: {type(e).__name__}: {e}",
                  file=sys.stderr, flush=True)
            if last_raw is not None:
                print(f"    [ERROR] raw output: {last_raw[:300]!r}", file=sys.stderr, flush=True)
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            traceback.print_exc()
            return dict(ERROR_RESULT)


def preflight_check():
    print("[INFO] Running preflight check against Gemini API...", flush=True)
    test = enrich_with_gemini(
        "Test Instructor", "APPI Matwork Level One, Two, Three. Ante & Post Natal Pilates.",
        "1 Test Street, TE5T 1AB", "https://example.com", retries=1,
    )
    if test.get("bio", "").startswith("[ENRICHMENT FAILED"):
        print("[FATAL] Preflight check failed — API unreachable or misconfigured. Aborting.",
              file=sys.stderr)
        sys.exit(1)
    print(f"[INFO] Preflight check passed (got: {test}).", flush=True)


def main():
    preflight_check()

    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
        fieldnames = list(rows[0].keys())

    for field in ("business_name", "base_qualification", "delivery_mode",
                  "insurance_confirmed", "bio"):
        if field not in fieldnames:
            fieldnames.append(field)

    for i, row in enumerate(rows):
        print(f"Enriching {i+1}/{len(rows)}: {row.get('name', '')[:40]}", flush=True)
        result = enrich_with_gemini(
            row.get("name", ""),
            row.get("description", ""),
            row.get("address", ""),
            row.get("website", ""),
        )
        row["business_name"] = result.get("business_name", "")
        row["base_qualification"] = result.get("base_qualification", "")
        row["delivery_mode"] = result.get("delivery_mode", "unknown")
        row["insurance_confirmed"] = result.get("insurance_confirmed", "unknown")
        row["bio"] = result.get("bio", "")
        time.sleep(1.0)  # stay under rate limits

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"\n[SUCCESS] Enriched {len(rows)} rows -> {OUTPUT_FILE}")
    needs_manual_bio = [r for r in rows if r["bio"].startswith("[ENRICHMENT FAILED")]
    if needs_manual_bio:
        print(f"[WARN] {len(needs_manual_bio)} row(s) need a manual bio — Gemini call failed:")
        for r in needs_manual_bio:
            print(f"    {r.get('name', '')}")


if __name__ == "__main__":
    main()

[INFO] Running preflight check against Gemini API...
[INFO] Preflight check passed (got: {'business_name': '', 'base_qualification': 'APPI Matwork Level One, Two, Three', 'delivery_mode': 'unknown', 'insurance_confirmed': 'unknown', 'bio': 'Test Instructor is a Pilates practitioner located at 1 Test Street, TE5T 1AB. They hold foundational qualifications in APPI Matwork Levels One, Two, and Three. Additionally, Test Instructor provides specialised ante and post natal Pilates instruction.'}).
Enriching 1/10: Agnieszka (Aga) Hanusiak (Fully Certifie
Enriching 2/10: Agnieszka Waszkielis (Fully Certified)
Enriching 3/10: Emilie Askew (Fully Certified)
Enriching 4/10: Helen Ainsworth (Fully Certified)
Enriching 5/10: Helen Pearce (Fully Certified)
Enriching 6/10: Jehian Yehia (Fully Certified)
Enriching 7/10: Jennifer Dennis (Fully Certified)
Enriching 8/10: Joanne Murphy (Fully Certified)
Enriching 9/10: Katrina Wade (Fully Certified)
Enriching 10/10: Noemi Nagy (Fully Certified)

[SUCCESS